# MERRA-2


<span style='font-family:serif'><font size="3.5"> Access to MERRA-2 data, streaming N granules to N local files. Subsetting takes place close to data. With data close to Client API, any subsequent aggregation, when possible, takes place close to the client. Reducing the amount of (remote) requests behind EDL authentication that Xarray makes. Furthermore, because subsetting takes place close to the data, then:

* Bytes are streamed as soon as they are processed by the server.
* Amount of data stream is much less than the entire file.


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import numpy as np
import matplotlib.pyplot as plt
# import pydap-specific tools
from pydap.client import open_url, get_cmr_urls 
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**    
    
We are interested in `MERRA-2` data, hourly data. MERRA-2 has several collections, organized by data products and frequency of data. In this case, we are interested in hourly data from the following collections:
    
* **doi**: 10.5067/QBZ6MG944HW0, **shortname**: M2I3NPASM. **Variables of interest**: air_temperature, surface_pressure, Ertel's Potential Vorticity, northward_wind, eastward_wind, specific_humidity, ozone_mass_mixing_ratio and (dimensions). Further information [click here]( https://disc.gsfc.nasa.gov/datasets/M2I3NPASM_5.12.4/summary).

* **doi**: 10.5067/HO9OVZWF3KW2, **shortname**: M2I3NVCHM, **Variables of interest**: Carbon Monoxide, O3 (and dimensions data). Further information [click here](https://disc.gsfc.nasa.gov/datasets/M2I3NVCHM_5.12.4/summary).


In [ ]:
MERRA2_M2I3NPASM_doi = "10.5067/QBZ6MG944HW0" # 
MERRA2_M2I3NPASM_ccid = "C1276812879-GES_DISC"

MERRA2_M2I3NVCHM_doi = "10.5067/HO9OVZWF3KW2"
MERRA2_M2I3NVCHM_ccid = "C1276812901-GES_DISC"

year_start = 2024
year_end = 2025


time_ranges = [[dt.datetime(year, 9, 21), dt.datetime(year, 10, 22)] for year in range(year_start, year_end)]

cmr_urls1 = [urls for time in time_ranges for urls in get_cmr_urls(doi=MERRA2_M2I3NPASM_doi, time_range=time, limit=1000)] # you can incread the limit of results
cmr_urls2 = [urls for time in time_ranges for urls in get_cmr_urls(doi=MERRA2_M2I3NVCHM_doi, time_range=time, limit=1000)]

print("################################################ \n We found a total of ", len(cmr_urls1)+len(cmr_urls2), "OPeNDAP URLS!!!\n################################################")

In [ ]:
len(cmr_urls1), len(cmr_urls2)

In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = auth.get_session()

In [ ]:
%%time
pyds1 = open_url(cmr_urls1[0], protocol="dap4", session=my_session)
pyds1.tree()

In [ ]:
%%time
pyds2 = open_url(cmr_urls2[0], protocol="dap4", session=my_session)
pyds2.tree()

In [ ]:
# output data directory
output_path = "data/"

Variables1 = ["/U", "/V", "/T","/PS"]
dims1 = list(set([dim for var in pyds1 for dim in pyds1[var].dims]))
Variables1 += dims1
print("Download only Vars in 1: ", Variables1)

Variables2 = ["/CO", "/O3"]
dims2 = list(set([dim for var in pyds2 for dim in pyds2[var].dims]))
Variables2 += dims2
print("Download only Vars in 2: ", Variables2)


## Subset by lat/lon regions


<span style='font-family:serif'><font size="3.5"> In addition to subsetting by Variable names, We are interested in a region of the NorthAtlantic Ocean delimited by:


In [ ]:
lat_min, lat_max = 15, 65
lon_min, lon_max = -75, 15

## Load coordinate data 

We only need one single file per collection.

In [ ]:
%%time
ds1 = xr.open_dataset(cmr_urls1[0].replace("https", "dap4"), engine="pydap", session=my_session)

In [ ]:
%%time
ds2 = xr.open_dataset(cmr_urls2[0].replace("https", "dap4"), engine="pydap", session=my_session)

### check that coordinate data is equalacross collections

<span style='font-family:serif'><font size="3.5"> This is a necessary step, to check if same subset can be applied to data from both collections


In [ ]:
Lon1, Lat1 = ds1['lon'], ds1['lat']
Lon2, Lat2 = ds2['lon'], ds2['lat']


iLon1 = np.where((Lon1>lon_min)&(Lon1 < lon_max))[0]
iLon2 = np.where((Lon2>lon_min)&(Lon2 < lon_max))[0]

iLat1 = np.where((Lat1>lat_min)&(Lat1 < lat_max))[0]
iLat2 = np.where((Lat2>lat_min)&(Lat2 < lat_max))[0]

if all(iLon1==iLon2) and all(iLat1==iLat2):
    iLon = iLon1
    iLat = iLat2
    print("Both collections belong to same Level 3 model output")
else:
    print('Data is not Level 3')


In [ ]:
dim_slices = {
    'lon': (iLon[0], iLon[-1]), 
    'lat': (iLat[0], iLat[-1]),
}

# Stream data into local directory

In [ ]:
%%time
dap_to_netcdf(cmr_urls1, session=my_session, keep_variables=Variables1, dim_slices=dim_slices, output_path=output_path)

In [ ]:
%%time
dap_to_netcdf(cmr_urls2, session=my_session, keep_variables=Variables2, dim_slices=dim_slices, output_path=output_path)

## Inspect the data

In [ ]:
ds1 = xr.open_mfdataset(output_path+"/MERRA2_400.inst3_3d_asm_*.nc4")
ds1

In [ ]:
ds2 = xr.open_mfdataset(output_path+"/MERRA2_400.inst3_3d_chm_*.nc4")
ds2